<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/cticonnect_retrieval_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# [CELL 1 - Setup Environment & Core Libraries]
!pip install -q sentence-transformers faiss-cpu

import os
import sys
import json
import random
import numpy as np
import networkx as nx
from typing import List, Dict, Tuple, Any

import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
import faiss

# Set deterministic seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CTIConnect Environment Initialized on Device: {device}")

CTIConnect Environment Initialized on Device: cuda


In [9]:
# [CELL 2 - Heterogeneous CTI Knowledge Base Construction]

class HeterogeneousCTIKnowledgeBase:
    """
    Constructs a unified CTI Knowledge Graph linking structured entities:
    CVE (Vulnerability) -> CWE (Weakness) -> CAPEC (Attack Pattern) -> MITRE ATT&CK (Technique)
    and indexes unstructured threat intelligence reports.
    """
    def __init__(self):
        self.kg = nx.DiGraph()
        self.reports = []
        self._build_mock_cti_graph()
        self._build_unstructured_reports()

    def _build_mock_cti_graph(self):
        # Adding Structured Entities
        self.kg.add_node("CVE-2023-38606", type="CVE", desc="Kaspersky Operation Triangulation kernel memory privilege escalation.")
        self.kg.add_node("CWE-123", type="CWE", desc="Write-what-where condition leading to memory corruption.")
        self.kg.add_node("CAPEC-124", type="CAPEC", desc="Shared Memory Manipulation.")
        self.kg.add_node("T1068", type="ATT&CK", desc="Exploitation for Privilege Escalation.")

        # Cross-source relationship edges
        self.kg.add_edge("CVE-2023-38606", "CWE-123", relation="EXPLOITS_WEAKNESS")
        self.kg.add_edge("CWE-123", "CAPEC-124", relation="MAPPED_TO_ATTACK")
        self.kg.add_edge("CAPEC-124", "T1068", relation="REALIZED_BY_TECHNIQUE")

    def _build_unstructured_reports(self):
        self.reports = [
            {"id": "Report_01", "text": "Operation Triangulation utilized CVE-2023-38606 to bypass iOS kernel memory mitigations and achieve privilege escalation (T1068)."},
            {"id": "Report_02", "text": "APT29 deployed custom malware targeting unpatched Exchange web servers via remote code execution vulnerabilities."},
            {"id": "Report_03", "text": "Shared memory manipulation vulnerabilities (CWE-123) allow unauthorized local execution threads to modify privileged buffers."}
        ]

    def schema_aware_traverse(self, cve_id: str) -> List[str]:
        """Performs structured schema traversal across the CTI Knowledge Graph."""
        if cve_id not in self.kg:
            return []
        visited_nodes = []
        curr = cve_id
        visited_nodes.append(f"{curr} [{self.kg.nodes[curr]['type']}]")

        # Traverse downstream relationships
        for neighbor in nx.descendants(self.kg, cve_id):
            visited_nodes.append(f"{neighbor} [{self.kg.nodes[neighbor]['type']}]")
        return visited_nodes

cti_kb = HeterogeneousCTIKnowledgeBase()
print(f"Indexed CTI Knowledge Graph with {cti_kb.kg.number_of_nodes()} entities and {len(cti_kb.reports)} threat reports.")

Indexed CTI Knowledge Graph with 4 entities and 3 threat reports.


In [10]:
# [CELL 3 - Dense Vector & Graph-Augmented CTI Retriever]

class CTIConnectRetriever:
    """
    Implements and contrasts:
    1. Generic Naive Dense Vector RAG
    2. CTIConnect Domain-Specific Structured Graph-Augmented Retrieval
    """
    def __init__(self, kb: HeterogeneousCTIKnowledgeBase):
        self.kb = kb
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        self._build_vector_index()

    def _build_vector_index(self):
        self.report_texts = [r["text"] for r in self.kb.reports]
        embeddings = self.embedder.encode(self.report_texts, convert_to_tensor=False)

        self.dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(self.dim)
        self.index.add(np.array(embeddings).astype('float32'))

    def naive_dense_retrieve(self, query: str, top_k: int = 2) -> List[str]:
        query_vec = self.embedder.encode([query]).astype('float32')
        _, indices = self.index.search(query_vec, top_k)
        return [self.report_texts[i] for i in indices[0]]

    def cti_structured_retrieve(self, cve_id: str, query: str) -> Dict[str, Any]:
        # Step 1: Structured schema traversal on KG
        graph_evidence = self.kb.schema_aware_traverse(cve_id)
        # Step 2: Dense retrieval over unstructured reports
        unstructured_evidence = self.naive_dense_retrieve(query, top_k=1)

        return {
            "structured_chain": " -> ".join(graph_evidence),
            "unstructured_reports": unstructured_evidence
        }

retriever = CTIConnectRetriever(cti_kb)
print("CTIConnect Dual Retrieval Engine compiled successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

CTIConnect Dual Retrieval Engine compiled successfully.


In [11]:
# [CELL 4 - Qualitative Retrieval Demonstration: Naive Dense RAG vs. CTI Structured Retrieval]

sample_query = "Map CVE-2023-38606 to its corresponding MITRE ATT&CK technique and exploit mechanism."

print("=========================================================")
print("CTICONNECT QUALITATIVE RETRIEVAL BENCHMARK")
print("=========================================================")

# 1. Naive Dense RAG Performance
naive_results = retriever.naive_dense_retrieve(sample_query, top_k=2)
print("\n[NAIVE DENSE RAG EVIDENCE RETRIEVAL]")
for idx, doc in enumerate(naive_results):
    print(f"  Doc {idx+1}: {doc}")

# 2. Structured Graph-Augmented CTI RAG Performance
structured_results = retriever.cti_structured_retrieve("CVE-2023-38606", sample_query)
print("\n[CTICONNECT STRUCTURED GRAPH-AUGMENTED RETRIEVAL]")
print(f"  Structured Knowledge Chain : {structured_results['structured_chain']}")
print(f"  Matched Threat Report      : {structured_results['unstructured_reports'][0]}")

print("=========================================================")

CTICONNECT QUALITATIVE RETRIEVAL BENCHMARK

[NAIVE DENSE RAG EVIDENCE RETRIEVAL]
  Doc 1: Operation Triangulation utilized CVE-2023-38606 to bypass iOS kernel memory mitigations and achieve privilege escalation (T1068).
  Doc 2: APT29 deployed custom malware targeting unpatched Exchange web servers via remote code execution vulnerabilities.

[CTICONNECT STRUCTURED GRAPH-AUGMENTED RETRIEVAL]
  Structured Knowledge Chain : CVE-2023-38606 [CVE] -> CWE-123 [CWE] -> T1068 [ATT&CK] -> CAPEC-124 [CAPEC]
  Matched Threat Report      : Operation Triangulation utilized CVE-2023-38606 to bypass iOS kernel memory mitigations and achieve privilege escalation (T1068).
